# URLvestigia — quickstart

**Run All.** Nothing to edit, nothing to install by hand, no API keys, and no
terminal at any point.

This notebook does one full pass of what the accelerator is for: it checks what
this network can reach, asks a question, keeps the answer *and the way the answer
was found* as a governed row — and then starts the dashboard and hands you a link
to it.

About a minute end to end; the preflight is the slow part. It writes one thing,
`data/urlvestigia.db`, which is gitignored, and it installs anything it needs into
this kernel as it goes.

In [ ]:
# --- bootstrap ---
# A notebook has no __file__, so the working directory is the only anchor there is.
# It is usually the notebook's own directory — but VS Code's jupyter.notebookFileRoot
# can point a kernel at the workspace folder instead, and `jupyter lab` started from
# elsewhere inherits wherever it was started. So search upward for the repository
# rather than assuming, and say so plainly when it is not there.
import sys
from pathlib import Path


def find_root(start=None):
    """The URLvestigia repository at or above `start`."""
    here = Path(start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        # Two markers, not one: scripts/new-accelerator.sh copies METADATA.yaml into
        # a fresh accelerator, so on its own it would match the wrong repository.
        if (candidate / "METADATA.yaml").is_file() and (candidate / "data" / "db.py").is_file():
            return candidate
    raise RuntimeError(
        f"No URLvestigia repository at or above {here}. "
        "Open quickstart.ipynb from inside the cloned repository.")


ROOT = find_root()
# The layers are directories, not installed packages, so they go on the path the
# same way app/server.py puts them there at runtime. ROOT itself is included so
# `from app import hosting` resolves — the same reason tests/conftest.py adds it.
for layer in ("retrieval", "data", "scripts", "."):
    path = str((ROOT / layer).resolve())
    if path not in sys.path:
        sys.path.insert(0, path)

print(f"repository  {ROOT}")
print(f"kernel      {sys.executable}")


## 1. Dependencies

The search library is one package. If this kernel does not have it, this cell
installs it — into *this* interpreter, which is frequently not the `pip` on your
PATH, and is the single most common reason a "but I installed it" notebook fails.

If the install cannot be done for you, nothing raises: the cell says what to run,
and every cell below reports itself skipped instead of throwing a traceback.

In [ ]:
import importlib
import importlib.util
import subprocess


def ensure(modules, requirements, what):
    """Make sure `modules` are importable here, installing `requirements` if not.

    Returns True when they are usable. Never raises: a quickstart that dies on its
    dependency cell has failed at the one job it has.
    """
    missing = [m for m in modules if importlib.util.find_spec(m) is None]
    if not missing:
        return True

    print(f"Installing {what} ({', '.join(missing)} missing)...")
    install = [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)]
    # A managed runtime image — a Cloudera AI session, say — usually has a
    # site-packages the session user cannot write to, and there the install has to
    # go to the user site instead. Not offered inside a virtualenv, where pip
    # rejects --user outright because the user site is not on its path.
    attempts = [install] if sys.prefix != sys.base_prefix else [install, install + ["--user"]]
    for attempt in attempts:
        try:
            subprocess.check_call(attempt)
            break
        except (subprocess.CalledProcessError, OSError) as exc:
            failure = exc
    else:
        print(f"  Could not install it automatically ({failure}). Run this yourself:\n")
        print(f"    {sys.executable} -m pip install -r {requirements}\n")
        print("  Then restart the kernel and Run All again.")
        return False

    # A package installed after the interpreter started is invisible until the
    # import system is told to look again.
    importlib.invalidate_caches()
    still_missing = [m for m in missing if importlib.util.find_spec(m) is None]
    if still_missing:
        print(f"  Installed, but {', '.join(still_missing)} is still not importable.")
        print("  Restart the kernel and Run All again.")
        return False
    print("  Installed.")
    return True


READY = ensure(["ddgs"], ROOT / "retrieval" / "requirements.txt", "the search library")
if READY:
    print("ready")


## 2. What can this network actually reach?

The same preflight `make doctor` runs, probing each corpus and **each web engine
individually** — a single blocked engine is worth seeing by name rather than
hidden inside the pool.

This is the slow cell: 10–40 seconds, longer if something is timing out. A blocked
engine here is a measurement of this network, not a defect. The next cell picks a
provider from whatever answered, so a locked-down network gets a readable
diagnosis instead of a failed search three cells later.

In [ ]:
if READY:
    import doctor

    probes = doctor.rows()
    for label, status, detail, seconds in probes:
        print(f"  {doctor.MARK[status]}  {label:16} {seconds:5.1f}s  {detail}")

    healthy = [label for label, status, _, _ in probes if status == doctor.OK]
    ENGINES = [label.removeprefix("web: ") for label in healthy if label.startswith("web: ")]
    APIS = [label for label in healthy if not label.startswith("web: ")]

    # Prefer the open web when any engine answered; otherwise search a corpus this
    # network can actually reach, rather than demonstrating a failure.
    PROVIDER = "ddgs" if ENGINES else (APIS[0] if APIS else None)
    if PROVIDER is None:
        READY = False
        print("\n  Nothing is reachable from here. Check the network, a proxy, or a")
        print("  TLS-intercepting firewall; the rows above name which it is.")
    else:
        print(f"\n  Searching with: {PROVIDER}"
              + (f" via {', '.join(ENGINES)}" if ENGINES else ""))
else:
    print("skipped - see the dependency cell above")


## 3. One question, recorded

Edit `QUERY` and re-run this cell as often as you like; every run adds a row.

The part worth watching is what gets *stored*. Options this corpus does not apply
are recorded as `NULL`, not as the value passed in — so the record can never claim
a filter that never ran. That single rule is what makes the table defensible months
later, and it is enforced in one place, `data/record.py`, for the dashboard, the
terminal, and this notebook alike.

In [ ]:
QUERY = "GLP-1 receptor agonist adverse event reporting"   # <- edit me

if READY:
    import db
    import record
    import urlvestigia

    # The store has to exist before the first row, exactly as app/server.py
    # ensures at import. Idempotent, and it migrates an older store in place.
    db.init_db()

    try:
        result = record.run(QUERY, provider=PROVIDER, max_results=10, backend=ENGINES)
    except urlvestigia.EngineError as exc:
        # Every engine failed and each said why. The network can go down between
        # the preflight and this cell, so this is not belt-and-braces.
        print(f"{record.label(PROVIDER)} search failed - no engine answered:")
        for engine, reason in exc.failures:
            print(f"  {engine}: {reason}")
        result = None
    except Exception as exc:
        print(f"{record.label(PROVIDER)} search failed - {exc}")
        result = None

    if result and result["urls"]:
        print(f'{len(result["urls"])} urls, recorded as search #{result["search_id"]}:\n')
        for url in result["urls"]:
            print(f"  {url}")
    elif result:
        # Not an error: the corpus answered and had nothing.
        print(f'No results. {record.label(PROVIDER)} had nothing for "{QUERY}".')
        print("Try editing QUERY above - a scholarly corpus will not match a")
        print("navigational web query, and vice versa.")
else:
    print("skipped - see the cells above")


## 4. The record

The deliverable. Every row carries the question and the options that produced it,
so the search can be reproduced or audited by someone who was not here.

Read the option columns carefully — they say two different things:

* **`n/a`** — this corpus has no such option. Nothing was filtered.
* **`any`** — it has one, and this search chose not to use it.

Collapsing those two is exactly the overstatement the record exists to prevent,
which is also why this table is hand-rendered rather than handed to pandas: a
DataFrame prints both as a blank cell.

In [ ]:
import html


def render(rows):
    """The recorded searches as a table. NULL and unset must not look alike."""
    def cell(value):
        if value is None:
            return '<span style="opacity:.5">n/a</span>'
        return html.escape(str(value)) if value != "" else '<span style="opacity:.5">any</span>'

    columns = ("id", "created_at", "provider", "region", "safesearch",
               "timelimit", "backend", "max_results")
    head = "".join(f'<th style="text-align:left;padding:4px 10px">{c}</th>'
                   for c in ("query", *columns, "urls"))
    body = ""
    for row in rows:
        cells = "".join(f'<td style="padding:4px 10px;vertical-align:top">{cell(row[c])}</td>'
                        for c in columns)
        body += (f'<tr style="border-top:1px solid rgba(128,128,128,.35)">'
                 f'<td style="padding:4px 10px;max-width:22em">{html.escape(row["query"])}</td>'
                 f'{cells}'
                 f'<td style="padding:4px 10px;text-align:right">{len(row["urls"])}</td></tr>')
    return (f'<table style="border-collapse:collapse;font-size:.9em">'
            f'<tr style="border-bottom:2px solid rgba(128,128,128,.6)">{head}</tr>'
            f'{body}</table>')


if READY:
    from IPython.display import HTML, display

    import db

    display(HTML(render(db.list_searches(limit=10))))
    print("n/a = this corpus has no such option.  any = it has one, unused.")
    # If you would rather have a DataFrame, and can live with it blurring the two:
    # import pandas as pd; pd.DataFrame(db.list_searches(limit=10))
else:
    print("skipped - see the cells above")


## 5. Export it

One row per URL, with the provenance of its search denormalized onto it — the shape
a reviewer sorts by domain and filters by provider, and the same shape the lakehouse
curates into. `NULL` is spelled out because CSV has only one kind of empty cell and
this record needs two.

This cell only *shows* the CSV, so Run All never leaves a file behind. The command
underneath writes the real one.

In [ ]:
if READY:
    import io

    import cli

    preview = io.StringIO()
    cli.write_csv(cli.export_rows(limit=10), preview)
    for line in preview.getvalue().splitlines()[:8]:
        print(line[:160])

    print("\nTo write the file itself:")
    print('    python scripts/cli.py export --format csv --out review-appendix.csv')
else:
    print("skipped - see the cells above")


## 6. Open the dashboard

The same record, in a browser — the interface for people who will never open a
terminal. Running this cell starts the server in the background and prints a link;
the search you ran above is already in it.

Three things worth knowing.

* It **reuses a server already on the port** rather than fighting it for one, so
  this is safe to run twice, and safe if your editor already started one.
* It keeps running after the cell finishes — that is the point — so section 7 is
  how you stop it.
* **In a Cloudera AI session it binds differently.** Your browser is outside the
  container, so `127.0.0.1` would point at your own laptop and connect to nothing.
  There the server binds every interface on `CDSW_APP_PORT` and the link becomes
  the session's proxied address. On a laptop it stays on loopback — this app has no
  authentication, and putting it on the local network is the one thing
  [the README](README.md#prerequisites) tells you not to do. The rule lives in
  [`app/hosting.py`](app/hosting.py).

In [ ]:
# --- dashboard ---
import atexit
import socket
import time

from IPython.display import HTML, display

from app import hosting

PORT = hosting.port()     # 8000 on a laptop; CDSW_APP_PORT in a Cloudera session
URL = hosting.url()       # loopback, or the session's proxied subdomain


def serving(port=PORT, host="127.0.0.1"):
    """Is something already answering on this port?

    Always asked over loopback even when the server is bound to every interface:
    this runs in the same container, and the question is whether the port is held,
    not whether the outside world can get to it.
    """
    with socket.socket() as probe:
        probe.settimeout(0.4)
        return probe.connect_ex((host, port)) == 0


def start_dashboard(wait_s=25):
    """Run the Serve layer in the background and wait for it to bind the port.

    Popen, not uvicorn.run(): the server has to outlive the cell, and a blocking
    call here would hang the kernel with no way to reach the browser. No --reload
    either — the reloader runs the server in a grandchild process, which would
    survive terminate() and leave the port held after the notebook says it stopped.
    """
    process = subprocess.Popen(
        [sys.executable, *hosting.uvicorn_argv()],
        cwd=str(ROOT), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    # Closing the notebook must not leave a server holding the port forever.
    atexit.register(process.terminate)

    deadline = time.monotonic() + wait_s
    while time.monotonic() < deadline:
        if serving():
            break
        if process.poll() is not None:  # it exited; the caller reports why
            break
        time.sleep(0.25)
    return process


DASHBOARD = globals().get("DASHBOARD")  # so re-running does not spawn a second one

if READY and ensure(["fastapi", "uvicorn", "jinja2"],
                    ROOT / "app" / "requirements.txt", "the dashboard"):
    # The process handle is better evidence than the port: a socket can still
    # answer for a moment after a server is told to stop, and trusting the port
    # alone would then report a dead link as "already running".
    mine = DASHBOARD is not None and DASHBOARD.poll() is None
    reused = not mine and serving()
    if not mine and not reused:
        DASHBOARD = start_dashboard()

    if serving():
        print("Already running - started by this notebook earlier." if mine else
              "Reusing the server already on this port." if reused else
              "Dashboard started.")
        print(f"  listening on {hosting.host()}:{PORT}")
        if hosting.reachable():
            display(HTML(
                f'<p style="font-size:1.1em">&#127760; '
                f'<a href="{URL}" target="_blank" rel="noopener">{URL}</a>'
                f'</p><p style="opacity:.7">Ctrl-click, or paste it into a browser.</p>'))
        else:
            # A session whose proxy address cannot be derived. The server is up and
            # correct; only the link is unknown, and saying which is the difference
            # between a five-minute fix and a bug report.
            print("\n  This is a Cloudera AI session, but CDSW_ENGINE_ID and")
            print("  CDSW_DOMAIN are not both set, so the proxied address cannot be")
            print("  derived here. The app is up and bound correctly - open it with")
            print("  the session's own web UI access for this port.")
    else:
        # Almost always an import error in the Serve layer. Ask for it directly
        # rather than making someone go and read a log that was sent to DEVNULL.
        print(f"The dashboard did not come up on port {PORT}.\n")
        check = subprocess.run([sys.executable, "-c", "import app.server"],
                               cwd=str(ROOT), capture_output=True, text=True)
        if check.returncode:
            print(check.stderr.strip()[-1000:])
        else:
            print(f"It imports cleanly, so port {PORT} is probably taken by")
            print("something else. Start it by hand to see what it says:")
            print(f"    {sys.executable} " + " ".join(hosting.uvicorn_argv()[1:]))
else:
    print("skipped - see the cells above")


## 7. Stop the dashboard

Left alone this does nothing, so Run All does not kill the server it just started.
Set the flag to `True` and run this cell when you are finished — or just shut the
kernel down, which stops it too.

In [ ]:
STOP_DASHBOARD = False   # <- set to True, then run this cell

process = globals().get("DASHBOARD")

if not STOP_DASHBOARD:
    if process is not None and process.poll() is None:
        print(f"Still running at {URL}")
        print("Set STOP_DASHBOARD = True above and run this cell to stop it.")
    else:
        print("Nothing was started from this notebook.")
elif process is None or process.poll() is not None:
    print("Nothing to stop.")
else:
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
    print("Dashboard stopped.")


## Where to go next

Everything above ran without a terminal. These are the same capabilities for when
you want one.

**The dashboard**, started by hand instead of by section 6:

```bash
make dev                                                # → http://127.0.0.1:8000/
python -m uvicorn app.server:app --reload --port 8000   # without make
```

**The terminal**, which writes the same records and pipes:

```bash
python scripts/cli.py search "your question" --provider arxiv -n 25
python scripts/cli.py list --urls
python scripts/cli.py export --format csv --out review-appendix.csv
```

**Before a live demo:** `make doctor` — the preflight from section 2, with a verdict.

**Deeper:**

* [`docs/EXAMPLE.md`](docs/EXAMPLE.md) — one question followed through all five layers
* [`retrieval/notebooks/eval.ipynb`](retrieval/notebooks/eval.ipynb) — how the engines
  and corpora actually compare on availability and overlap
* [`docs/ARCHITECTURE.md`](docs/ARCHITECTURE.md) — the decisions worth defending